In [6]:
# ============================================================================
# 03a-PATCH — fix empty responses, trim the web arm
# ============================================================================
def call(mid, user, max_tok, web=False, timeout=180):
    """One question, one API call. Empty completions are now RETRIED —
    gpt-oss returned an empty body once in the preflight and a silent gap
    is worse than a slow retry."""
    body = {"model":mid, "temperature":TEMP, "max_tokens":max_tok,
            "messages":[{"role":"system","content":PR["system"]},
                        {"role":"user","content":user}]}
    if web: body["plugins"] = [{"id":"web","max_results":3}]
    last_empty = None
    for att in range(5):
        if STOP.is_set():
            return dict(text=None,usage={},http=0,error="stopped",prov=None)
        try:
            r = requests.post(ENDPOINT, headers={"Authorization":f"Bearer {KEY}"},
                              json=body, timeout=timeout)
            if r.status_code == 200:
                d = r.json(); ch = d["choices"][0]
                txt = ch["message"].get("content")
                if not txt:
                    last_empty = dict(text=None, usage=d.get("usage",{}), http=204,
                                      error=f"empty ({ch.get('finish_reason')})",
                                      prov=d.get("provider"))
                    time.sleep(2 + att); continue          # <-- retry, don't give up
                return dict(text=txt, usage=d.get("usage",{}), http=200,
                            error=None, prov=d.get("provider"))
            if r.status_code in (429,503,529):
                time.sleep(min(5*(1.8**att),60)); continue
            if r.status_code in (401,402,403):
                STOP.set()
                return dict(text=None,usage={},http=r.status_code,
                            error=r.text[:200],prov=None)
            return dict(text=None,usage={},http=r.status_code,
                        error=r.text[:200],prov=None)
        except Exception:
            time.sleep(min(2**att,20))
    return last_empty or dict(text=None,usage={},http=0,
                              error="retries exhausted",prov=None)

# web arm runs on REAL questions only — decoys would add $1.60 of search fees
WEB_ITEMS = FULL[~FULL.is_decoy].reset_index(drop=True)
print(f"web arm trimmed: {len(WEB_ITEMS)} items (was {len(FULL)})")
print(f"  search fee now ~${len(WEB_ITEMS)*3*4/1000:.2f} (was "
      f"${len(FULL)*3*4/1000:.2f})")
main = 1.28; swap = 0.17
web  = len(WEB_ITEMS)*3*4/1000 + len(WEB_ITEMS)*(160/1e6*.1875 + 480/1e6*.9375)
print(f"\n  main ${main:.2f} + swap ${swap:.2f} + web ${web:.2f} "
      f"= ${main+swap+web:.2f}   (budget ${BUDGET})")
assert main+swap+web < BUDGET*0.9, "still too close to the budget"
print("  OK — comfortably inside budget")

web arm trimmed: 216 items (was 336)
  search fee now ~$2.59 (was $4.03)

  main $1.28 + swap $0.17 + web $2.70 = $4.15   (budget $5.0)
  OK — comfortably inside budget


In [5]:
# ============================================================================
# 03b — PREFLIGHT  (~2 cents; verifies everything before the real run)
# ============================================================================
MINI  = pd.concat([FULL[FULL.condition==c].head(3)
                   for c in ["habit_agrees","habit_misleads","decoy"]]).reset_index(drop=True)
MINIU = USUAL.head(3)
print(f"preflight: {len(MINI)} main + {len(MINIU)} usually per model "
      f"= ~{(len(MINI)+len(MINIU))*len(MODELS)+8} calls\n")

SPEND["usd"] = 0.0; STOP.clear()
rows = []

def probe(name, mid, pin, pout, mt, item, swap=False, web=False):
    t0 = time.time()
    res = call(mid, render(item, swap), mt, web, timeout=180)
    parsed, tier = parse(res["text"]) if res["http"]==200 else (None,"failed")
    pred = normalize(parsed)
    u = res.get("usage") or {}
    cost = u.get("prompt_tokens",0)/1e6*pin + u.get("completion_tokens",0)/1e6*pout
    with LOCK: SPEND["usd"] += cost
    return dict(model=name, cond=item["condition"], swap=swap, web=web,
        source=item["source"], target=item.get("target"), truth=item["answer"],
        pred=pred, correct=(pred==item["answer"]),
        conf=(parsed or {}).get("confidence"), tier=tier,
        rationale=(parsed or {}).get("rationale"),
        raw=(res["text"] or "")[:200], http=res["http"], err=res["error"],
        prov=res["prov"], out_tok=u.get("completion_tokens"),
        secs=time.time()-t0, cost=cost)

print("="*96); print("A. EVERY MODEL, EVERY QUESTION TYPE"); print("="*96)
for name, mid, pin, pout, mt in MODELS:
    print(f"\n{name}", flush=True)
    for _, it in pd.concat([MINI, MINIU]).iterrows():
        r = probe(name, mid, pin, pout, mt, it.to_dict()); rows.append(r)
        ok = r["http"]==200 and r["pred"]
        print(f"  [{'OK ' if ok else 'BAD'}] {r['cond']:<15}"
              f"{str(r['source']):>9}->{str(r['target'] or '—'):<9} "
              f"truth={str(r['truth']):<16} pred={str(r['pred']):<16}"
              f"{r['tier']:<9}{r['secs']:>5.1f}s"
              + (f"  ERR {str(r['err'])[:55]}" if r["http"]!=200 else ""))

print("\n"+"="*96); print("B. ORDER-SWAP — does flipping the options change the answer?")
print("="*96)
for name, mid, pin, pout, mt in MODELS[:3]:
    it = MINI[MINI.condition=="habit_agrees"].iloc[0].to_dict()
    a = probe(name, mid, pin, pout, mt, it, swap=False)
    b = probe(name, mid, pin, pout, mt, it, swap=True)
    rows += [a,b]
    print(f"  {name:<20} normal={str(a['pred']):<14} swapped={str(b['pred']):<14}"
          f"  {'same' if a['pred']==b['pred'] else 'DIFFERENT'}")

print("\n"+"="*96); print("C. WEB SEARCH — is retrieval actually happening?"); print("="*96)
for _, it in MINI.head(2).iterrows():
    r = probe("gemini-3.7-flash","google/gemini-3.7-flash",.1875,.9375,3000,
              it.to_dict(), web=True); rows.append(r)
    print(f"  [{'OK ' if r['http']==200 else 'BAD'}] {r['cond']:<15}"
          f"{r['source']}->{r['target']}  pred={str(r['pred']):<16}"
          f"{r['secs']:>5.1f}s  ${r['cost']:.4f}  {r['out_tok']} tok")
    print(f"        {str(r['rationale'])[:130]}")

T  = pd.DataFrame(rows)
ok = T[T.http==200]
print("\n"+"="*96); print("D. DID ANYTHING BREAK?"); print("="*96)
checks = [
 ("every model responded", T.groupby('model').http.apply(lambda s:(s==200).any()).all()),
 ("no HTTP failures",      (T.http==200).all()),
 ("everything parsed",     ok.pred.notna().all()),
 ("decoys answered",       len(ok[ok.cond=='decoy'])>0),
 ("'usually' probes work", len(ok[ok.cond=='usually'])>0),
 ("swap arm works",        ok[ok.swap].pred.notna().all() if ok.swap.any() else True),
 ("web arm works",         ok[ok.web].pred.notna().all() if ok.web.any() else True),
]
for n_,c in checks: print(f"  [{'PASS' if c else 'FAIL'}] {n_}")
if (T.http!=200).any():
    print("\n  FAILURES:")
    print("  "+T[T.http!=200][["model","cond","http","err"]].to_string(index=False).replace("\n","\n  "))
if ok.pred.isna().any():
    print("\n  UNPARSED — raw text:")
    for _, r in ok[ok.pred.isna()].head(3).iterrows():
        print(f"    {r.model}/{r.cond}: {r.raw!r}")

print("\n"+"="*96); print("E. SANITY (3 items each — machinery check, not a result)")
print("="*96)
print(ok.pivot_table(index="model", columns="cond", values="correct",
                     aggfunc="mean").round(2).to_string())
d = ok[ok.cond=="decoy"]
print("\ndecoy refusal rate (some models should be >0):")
print("  "+(d.assign(ref=d.pred=="not_established").groupby("model").ref.mean()
      .round(2).to_string().replace("\n","\n  ")))

print("\n"+"="*96); print("F. READ THESE RATIONALES"); print("="*96)
for m in ok.model.unique():
    s = ok[(ok.model==m)&(ok.cond=="habit_misleads")&(~ok.web)]
    if not len(s): continue
    r = s.iloc[0]
    print(f"\n  {m}  |  {r.source} -> {r.target}  (truth: {r.truth})")
    print(f"    said {r.pred}, confidence {r.conf}")
    print(f"    {str(r.rationale)[:150]}")

print("\n"+"="*96); print("G. COST PROJECTION"); print("="*96)
print(f"  preflight cost: ${SPEND['usd']:.4f} over {len(T)} calls\n")
per = ok[~ok.web].groupby("model").agg(secs=("secs","mean"), cost=("cost","mean"),
                                       out=("out_tok","mean"))
n_calls = len(FULL)+len(USUAL)
print(f"{'model':<20}{'s/call':>8}{'out tok':>9}{'$/call':>9}{'full run $':>12}{'min':>7}")
print("-"*66)
tot = 0
for m in per.index:
    c = per.loc[m,"cost"]*n_calls; tot += c
    print(f"{m:<20}{per.loc[m,'secs']:>8.1f}{per.loc[m,'out']:>9.0f}"
          f"{per.loc[m,'cost']:>9.4f}{c:>12.3f}"
          f"{per.loc[m,'secs']*n_calls/WORKERS/60:>7.0f}")
web_cost = len(FULL)*3*4/1000 + len(FULL)*(160/1e6*.1875 + 400/1e6*.9375)
print(f"\n  main arms        : ${tot:.2f}")
print(f"  swap (3 models)  : ${tot*0.13:.2f}")
print(f"  web (Gemini, x3) : ${web_cost:.2f}")
print(f"  PROJECTED TOTAL  : ${tot + tot*0.13 + web_cost:.2f}   (budget ${BUDGET})")

good = all(c for _,c in checks)
print("\n"+"="*96)
print("ALL CHECKS PASSED — cell 03c is safe to run" if good
      else "*** SOMETHING FAILED — do not run 03c ***")
print("="*96)

preflight: 9 main + 3 usually per model = ~68 calls

A. EVERY MODEL, EVERY QUESTION TYPE

gpt-oss-120b
  [OK ] habit_agrees       NFIL3->PTGS2     truth=inhibits         pred=inhibits        strict    18.7s
  [OK ] habit_agrees        DOK1->ITGB2     truth=inhibits         pred=not_established strict     5.3s
  [OK ] habit_agrees       HDAC1->ABCB1     truth=inhibits         pred=inhibits        strict     2.2s
  [OK ] habit_misleads     NFIL3->IL3       truth=stimulates       pred=inhibits        strict     3.1s
  [OK ] habit_misleads      DOK1->RASGRP3   truth=stimulates       pred=not_established strict     3.9s
  [OK ] habit_misleads     HDAC1->COL1A2    truth=stimulates       pred=inhibits        strict     6.9s
  [OK ] decoy             DNMT3B->OXT       truth=not_established  pred=not_established strict     3.1s
  [OK ] decoy             DNMT3B->STAT3     truth=not_established  pred=not_established strict     9.7s
  [OK ] decoy              MAPK7->HIPK2     truth=not_established

In [ ]:
# ============================================================================
# 03c — FULL COLLECTION   *** SPENDS MONEY — run only after 03b passes ***
# ----------------------------------------------------------------------------
# Resumable and safe to interrupt: re-running skips anything already answered.
# ============================================================================
SPEND["usd"] = 0.0; STOP.clear()
RUN_T0 = time.time()

print("="*70); print("MAIN ARM"); print("="*70)
for name, mid, pin, pout, mt in MODELS:
    if STOP.is_set(): print("\n*** HALTED ***"); break
    print(f"\n{name}", flush=True)
    run(name, mid, pin, pout, mt, FULL,  "main")
    run(name, mid, pin, pout, mt, USUAL, "usually")

print("\n"+"="*70); print("ORDER-SWAP ARM"); print("="*70)
MAIN_ONLY = FULL[~FULL.is_decoy]
for name, mid, pin, pout, mt in MODELS[:3]:
    if STOP.is_set(): break
    run(name, mid, pin, pout, mt, MAIN_ONLY, "swap", swap=True)

print("\n"+"="*70); print("WEB-SEARCH ARM"); print("="*70)
if not STOP.is_set():
    run("gemini-3.7-flash","google/gemini-3.7-flash",.1875,.9375,3000,
        FULL, "web", web=True)

print("\n"+"="*70); print("INVENTORY"); print("="*70)
A = pd.concat([pd.read_json(p, lines=True) for p in RES.glob("*.jsonl")],
              ignore_index=True)
print(f"rows {len(A):,}   spent ${SPEND['usd']:.2f}   "
      f"wall clock {(time.time()-RUN_T0)/60:.0f} min")
print(f"http failures {(A.http!=200).mean():.2%}   "
      f"unparsed {(A[A.http==200].pred.isna()).mean():.2%}")
print("\nunique items per model x arm:")
print(A[A.http==200].pivot_table(index="model", columns="arm",
      values="item_id", aggfunc="nunique", fill_value=0).to_string())
print("\nEXPECTED:  main 336 | usually 108 | swap 216 (3 models) | web 336 (Gemini)")
if (A.http!=200).any():
    print("\nfailures:"); print(A[A.http!=200].groupby(["model","http"]).size().to_string())
if (A[A.http==200].pred.isna()).any():
    u = A[(A.http==200) & A.pred.isna()]
    print(f"\nunparsed ({len(u)}):"); print(u.groupby("model").size().to_string())
    for _, r in u.head(3).iterrows(): print(f"    {r.model}: {str(r.raw)[:120]!r}")
print("\nspend by model:")
print(A.groupby("model").cost_usd.sum().round(3).to_string())

In [7]:
# ============================================================================
# 03-TRUTH — spot-check the ground truth before committing
# ----------------------------------------------------------------------------
# Every "habit_misleads" item claims a protein does the OPPOSITE of its usual
# behaviour. Those are exactly the records most likely to be curation errors.
# If they are wrong, our effect is measurement error, not model behaviour.
# This pulls the supporting PubMed IDs so we can check a sample by hand.
# ============================================================================
import urllib.request, urllib.parse
from pathlib import Path
import pandas as pd, numpy as np

ROOT = Path.cwd().parent
RAW  = ROOT/"data"/"raw"

# find the references behind each benchmark edge
ref_map = {}
for n in ["causal_ppi","collectri","kinaseextra"]:
    d = pd.read_parquet(RAW/f"{n}.parquet")
    col = "references" if "references" in d.columns else None
    if col is None: continue
    for s,t,r,src in zip(d.source_genesymbol, d.target_genesymbol,
                         d[col], d.get("sources", d[col])):
        if isinstance(r,str) and r.strip():
            ref_map.setdefault((s,t), set()).update(
                x.split(":")[-1] for x in r.split(";") if x.strip())

B = pd.read_json(ROOT/"data"/"processed"/"sighabit_v2.jsonl", lines=True)
B["pmids"] = [sorted(ref_map.get((r.source, r.target), set()))[:4]
              for _, r in B.iterrows()]
B["n_pmid"] = B.pmids.map(len)
print(f"items with at least one PubMed ID: {(B.n_pmid>0).mean():.0%}")

print("\n" + "="*92)
print("20 'HABIT MISLEADS' ITEMS TO VERIFY BY HAND")
print("="*92)
print("For each: does the cited paper really show this direction of effect?\n")
rng = np.random.RandomState(0)
S = B[(B.condition=="habit_misleads") & (B.n_pmid>0)]
S = S.iloc[rng.permutation(len(S))[:20]]
for i, (_, r) in enumerate(S.iterrows(), 1):
    print(f"{i:>3}. {r.source} -> {r.target}   OmniPath says: {r.answer.upper()}")
    print(f"     ({r.source} usually {r.protein_usually}; {r.n_refs} refs; {r.component})")
    for p in r.pmids[:3]:
        print(f"     https://pubmed.ncbi.nlm.nih.gov/{p}/")
    print()

print("="*92)
print("SAME FOR 'HABIT AGREES' — so we check both conditions equally")
print("="*92)
S2 = B[(B.condition=="habit_agrees") & (B.n_pmid>0)]
S2 = S2.iloc[rng.permutation(len(S2))[:10]]
for i, (_, r) in enumerate(S2.iterrows(), 1):
    print(f"{i:>3}. {r.source} -> {r.target}   OmniPath says: {r.answer.upper()}"
          f"   ({r.n_refs} refs)")
    for p in r.pmids[:2]:
        print(f"     https://pubmed.ncbi.nlm.nih.gov/{p}/")

B.to_json(ROOT/"data"/"processed"/"sighabit_v2_with_pmids.jsonl",
          orient="records", lines=True)
print(f"\nwritten: sighabit_v2_with_pmids.jsonl")
print("\nHOW TO USE THIS:")
print("  Open ~10 of the 'habit misleads' links and read the abstract.")
print("  Mark each: CORRECT / WRONG / AMBIGUOUS.")
print("  If 8+ of 10 are correct, the labels are sound and we proceed.")
print("  If 3+ are wrong, we must audit the whole set before publishing.")

items with at least one PubMed ID: 100%

20 'HABIT MISLEADS' ITEMS TO VERIFY BY HAND
For each: does the cited paper really show this direction of effect?

  1. PAX3 -> MBP   OmniPath says: INHIBITS
     (PAX3 usually stimulates; 2 refs; collectri)
     https://pubmed.ncbi.nlm.nih.gov/22532290/
     https://pubmed.ncbi.nlm.nih.gov/7546735/

  2. PTPN6 -> ABL1   OmniPath says: STIMULATES
     (PTPN6 usually inhibits; 6 refs; causal_ppi)
     https://pubmed.ncbi.nlm.nih.gov/10488096/
     https://pubmed.ncbi.nlm.nih.gov/10574931/
     https://pubmed.ncbi.nlm.nih.gov/12468540/

  3. DVL1 -> APC   OmniPath says: INHIBITS
     (DVL1 usually stimulates; 3 refs; causal_ppi)
     https://pubmed.ncbi.nlm.nih.gov/10330181/
     https://pubmed.ncbi.nlm.nih.gov/11425858/
     https://pubmed.ncbi.nlm.nih.gov/20224554/

  4. HDAC1 -> COL1A2   OmniPath says: STIMULATES
     (HDAC1 usually inhibits; 2 refs; collectri)
     https://pubmed.ncbi.nlm.nih.gov/16464847/
     https://pubmed.ncbi.nlm.nih.gov/2